# Chatbot preoperatorio
## Inteligencia artificial generativa


Objetivo: Crear una Inteligencia artificial que sea capaz de dar informacion medica pre operatoria para la seguridad de los pacientes, siguiendo restricciones y teniendo un registro de las conversaciones.

Tecnologias utilizadas: 
    Python 3.11.9: Lenguaje principal para el desarrollo

    Langchain: Frameworks para la construccion del 
               agente de IA

    OpenAI: Modelo utilizado GPT -4.1

    Jupyter Notebook: Entorno de desarrollo

    Pandas: Manipulacion de datos y almacenamiento

    Psutil: Obtencion de métricas

    Streamlit: Desarrollo de dashboard

    CSV: Almacenamiento de logs y metricas del chatbot


Arquitectura:

                           Usuario
                               │
                               ▼
                  Interfaz (Jupyter Notebook)
                               │
                               ▼
                    Validación de la consulta
                  (Seguridad y datos sensibles)
                               │
                               ▼
                  Recuperación de contexto (RAG)
                  Base de conocimiento interna
                               │
                               ▼
                 Construcción del Prompt Final
        (Prompt del sistema + contexto + consulta)
                               │
                               ▼
                    Agente LangChain
                               │
          ┌────────────────────┼────────────────────┐
          ▼                    ▼                    ▼
    Buscar recomendaciones   Generar reporte   Evaluar riesgo
          │                    │                    │
          └────────────────────┼────────────────────┘
                               ▼
                     Modelo de Lenguaje (GPT)
                               │
                               ▼
                  Generación de la respuesta
                               │
                               ▼
                Registro de observabilidad
          (latencia, CPU, RAM, errores y logs)
                               │
                               ▼
                  Respuesta al usuario

## Instalacion

pip install langchain

pip install langchain-openai

pip install langchain-community

pip install pandas

pip install psutil

pip install streamlit

pip install python-dotenv

### Imports

In [2]:
import os
import time
import psutil
import pandas as pd


from datetime import datetime

from dotenv import load_dotenv

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent


print("Ejecucion exitosa")

C:\Users\ferna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Ejecucion exitosa


## Variables de entorno

In [ ]:
load_dotenv()

#Ingresar token de github para correcto funcionameinto

print("GITHUB_TOKEN:", os.getenv("GITHUB_TOKEN"))
print("OPENAI_BASE_URL:", os.getenv("OPENAI_BASE_URL"))

llm= ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4.1",
    temperature=0,
    streaming=True
)

print("Ejecucion exitosa")

NameError: name 'load_dotenv' is not defined

## Prompt

In [4]:
system_prompt="""
        Eres un asistente de una clinica el cual esta especilizado en recomendaciones preoperatorias. 
        Tienes que seguir el siguiente reglamento:
            -Solo dar recomendaciones generales.
            -NO puedes recomendar medicamentos ni dosis.
            -NO realizar diagnosticos.
            -Si te piden realizar un diagnostico debes rechazarlo y explicar el pq no debes.
        
        Tu funcion principal es orientar a los pacientes antes de una cirugia utilizando las herramientas
        disponibles cuando sean necesarias y Nunca inventes información médica.
    """

print("ejecucion del prompt exitoso")

ejecucion del prompt exitoso


## Base de conocimiento

In [5]:
Base_conocimientos={
    "ayuno": 
    """Se recomienda ayuno de 6 a 8 horas antes de una cirugía.""",

    "medicamentos": 
    """Debe consultar con su médico antes de suspender cualquier medicamento.""",

    "alergias": 
    """Es importante informar al equipo médico sobre cualquier alergia.""",

    "higiene": 
    """Se recomienda bañarse antes del procedimiento según indicaciones médicas.""",

    "alcohol": 
    """Evitar consumo de alcohol al menos 24 horas antes de la cirugía.""",

    "tiempo": 
    """Se recomienda llegar con aticipasion al centro medico"""
}

print("Base de conocimiento ejecutada")

Base de conocimiento ejecutada


## RAG

In [6]:
def buscar_contexto(pregunta):
    contexto = []
    pregunta = pregunta.lower()

    for clave, valor in Base_conocimientos.items():
        if clave in pregunta:
            contexto.append(valor)

        if not contexto:
            contexto.append("""Siga siempre las indicaciones generales de su equipo medico""")
        return contexto
    
print("RAG implementado")

RAG implementado


## Planificador

In [7]:
def planificar_tarea(consulta):
    consulta = consulta.lower()

    if "reporte" in consulta:
        return [
            "Buscar informacion",
            "Generar reporte",
            "Entregar resultado"
        ]

    elif "riesgo" in consulta:
        return [
            "Analizar antecedentes",
            "Evaluar riesgo",
            "Entregar evaluacion"
        ]

    elif "recordatorio" in consulta:
        return [
            "Registrar recordatorio",
            "Generar respuesta"
        ]
    return [
        "Buscar recomendaciones",
        "Generar respuesta"
    ]

print("Planificador implementado")

Planificador implementado


## Herramientas (tools)

In [8]:
@tool
def buscar_recomendacion(pregunta:str) -> str:
    """
    Busca recomendaciones medicas preoperatorias
    """

    docs = buscar_contexto(pregunta)

    return "\n".join(docs)

@tool
def generar_reporte(texto:str)-> str:
    """
    Genera un reporte resmido
    """

    return f"""
    REPORTE PREOPERATORIO
    {texto}
    """

@tool
def evaluar_riesgo(texto:str)-> str:
    """
    Evaluá riesgos simples.
    """

    texto = texto.lower()

    if "alcohol" in texto:
        return "Posible incumplimiento de indicaciones preoperatorias"
    
    if "alergia" in texto:
        return "Se detectan anteedentes de alergias"
    
    return "No se identifican riesgos relevantes."

@tool
def registro_recordatorio(texto:str):
    """
    Registra recordatorios para el paciente
    """

    return f"recordatorio registrado {texto}"

print("Tools implementadas")

Tools implementadas


## Creacion de agente

In [9]:
agent = create_agent(
    model=llm,
    tools = [
        buscar_recomendacion,
        generar_reporte,
        evaluar_riesgo,
        registro_recordatorio
    ])

print("Agente creado")

Agente creado


## Observabilidad

In [10]:
LOG_FILE ="logs_chatbot.csv"

def registrar_log(
        pregunta,
        respuesta,
        latencia,
        cpu,
        ram,
        error=False
):
    fila = {
        "fecha": datetime.now(),
        "pregunta": pregunta,
        "respuesta":respuesta,
        "latencia":latencia,
        "cpu":cpu,
        "ram":ram,
        "error":error
    }

    df = pd.DataFrame([fila])

    try:
        df.to_csv(
            LOG_FILE,
            mode="a",
            index=False,
            header=False
        )

    except:
        df.to_csv(
            LOG_FILE,
            index=False
        )
    

def obtener_metricas():
    return {
        "cpu": psutil.cpu_percent(),
        "ram": psutil.virtual_memory().percent
    }

## Definicion de datos sensibles (seguridad)

In [11]:
datos_sensibles = [
    "rut",
    "contraseña",
    "clave",
    "tarjeta"
]

print("Datos sensibles definidos")

Datos sensibles definidos


## Historial

In [12]:
historial_chat=[]

print("Historial creado")

Historial creado


## Chatbot

In [13]:
def ChatBot_Medico3():
    print("--------- ChatBot preoperatorio Titirilquen ---------")
    print("Escribe 'salir' para terminar la conversación\\n")


    while True:

        pregunta = input("\nUsuario:")

        if pregunta.lower()=="salir":
            print("Gracias por utilizar: ChatBot clinica titirilquen")
            break

        if any(
            palabra in pregunta.lower()
            for palabra in datos_sensibles
        ):
            print("\n No puedo procesar informacion sensible")
            continue

        try:

            inicio = time.time()

            documentos = buscar_contexto(pregunta)

            contexto = "\n".join(documentos)

            plan = planificar_tarea(pregunta)

            plan_texto = "\n".join(plan)

            mensaje_usuario = f"""
            Plan de acción:
            {plan_texto}

            Contexto recuperado:
            {contexto}

            Consulta del paciente:
            {pregunta}
            """

            mensajes = historial_chat.copy()

            mensajes.append(
                {
                    "role": "user",
                    "content": mensaje_usuario
                }
            )

            resultado = agent.invoke(
                {
                    "messages": mensajes
                }
            )

            respuesta = resultado["messages"][-1].content

            historial_chat.append(
                {
                    "role": "assistant",
                    "content": respuesta
                }
            )

            fin = time.time()

            latencia = fin - inicio

            metricas = obtener_metricas()

            registrar_log(
                pregunta,
                respuesta,
                latencia,
                metricas["cpu"],
                metricas["ram"],
                False
            )

            print("\nAsistente\n")
            print(respuesta)

        except Exception as e:

            registrar_log(
                pregunta,
                str(e),
                0,
                0,
                0,
                True
            )

            print(f"\nError: {e}")

ChatBot_Medico3()  

--------- ChatBot preoperatorio Titirilquen ---------
Escribe 'salir' para terminar la conversación\n

Asistente

¡Hola! ¿En qué puedo ayudarte hoy? Si tienes alguna consulta médica o necesitas recomendaciones específicas, por favor dime más detalles para poder orientarte mejor. Recuerda siempre seguir las indicaciones de tu equipo médico.
Gracias por utilizar: ChatBot clinica titirilquen
